In [ ]:
import re, os
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import lsdb
from lsdb.streams import CatalogStream
from huggingface_hub import HfApi, hf_hub_download

REPO = "UniverseTBD/mmu_ssl_legacysurvey_north"
BASE = "mmu_ssl_legacysurvey_north/dataset/"     # main catalog, NOT _10arcs
api  = HfApi()

legacy_scalar = lsdb.open_catalog(
    f"hf://datasets/{REPO}",
    columns=["object_id", "ra", "dec"],
)
prov_scalar = lsdb.open_catalog(
    "hf://datasets/UniverseTBD/mmu_desi_provabgs",
    columns=["object_id", "ra", "dec", "LOG_MSTAR", "Z_HP", "AVG_SFR", "TAGE_MW", "Z_MW"],
)

xm = legacy_scalar.crossmatch(prov_scalar, radius_arcsec=0.1, n_neighbors=1)

batch, n = [], 0
for chunk in CatalogStream(catalog=xm):
    if len(chunk):
        batch.append(chunk); n += len(chunk)
    if n >= 2000:
        break

matches = pd.concat(batch).head(2000).reset_index()
matches.to_parquet("sample_2k_matches_metadata.parquet")
print(len(matches), matches.columns.tolist())

In [ ]:
print(matches["_dist_arcsec"].describe())

In [ ]:
pixels = legacy_scalar.get_healpix_pixels()
orders = sorted({p.order for p in pixels})
known  = {(p.order, p.pixel) for p in pixels}

def partition_of(hp29):
    for o in orders:
        key = (o, hp29 >> (2 * (29 - o)))
        if key in known:
            return key

matches["_part"] = matches["_healpix_29"].map(partition_of)
assert matches["_part"].notna().all(), "some rows mapped to no partition"
print(matches.groupby("_part").size())

In [ ]:
pat = re.compile(r"Norder=(\d+)/Dir=\d+/Npix=(\d+)\.parquet$")

path_by_pixel = {}
for f in api.list_repo_files(REPO, repo_type="dataset"):
    if not f.startswith(BASE):
        continue
    m = pat.search(f)
    if m:
        key = (int(m.group(1)), int(m.group(2)))
        assert key not in path_by_pixel, f"ambiguous: {f}"
        path_by_pixel[key] = f

info  = api.repo_info(REPO, repo_type="dataset", files_metadata=True)
sizes = {s.rfilename: s.size for s in info.siblings}

for part in matches["_part"].unique():
    p = path_by_pixel[part]
    assert "_10arcs" not in p
    print(part, p, round(sizes[p] / 1e9, 2), "GB")

In [ ]:
os.makedirs("cutouts", exist_ok=True)

# smallest file first, so failures surface cheaply
parts = sorted(matches["_part"].unique(), key=lambda p: sizes[path_by_pixel[p]])

for part in parts:
    wanted = set(matches.loc[matches["_part"] == part, "_healpix_29"])
    if all(os.path.exists(f"cutouts/{h}.npy") for h in wanted):
        print(part, "cached"); continue

    local = hf_hub_download(REPO, path_by_pixel[part],
                            repo_type="dataset", local_dir="hats_tmp")
    try:
        pf = pq.ParquetFile(local)
        for rg in range(pf.num_row_groups):
            idx = pf.read_row_group(rg, columns=["_healpix_29"])["_healpix_29"].to_pylist()
            hits = [i for i, h in enumerate(idx) if h in wanted]
            if not hits:
                continue
            imgs = pf.read_row_group(rg, columns=["image"])["image"].to_pylist()
            for i in hits:
                np.save(f"cutouts/{idx[i]}.npy", np.asarray(imgs[i]))
            del imgs
        del pf
    finally:
        os.remove(local)

    found = sum(1 for h in wanted if os.path.exists(f"cutouts/{h}.npy"))
    print(part, f"— {found}/{len(wanted)}")
    assert found > 0, f"{part}: matched no rows"

In [ ]:
meta = pd.read_parquet("sample_2k_matches_metadata.parquet").set_index("_healpix_29", drop=False)
obj = np.load(f"cutouts/{meta['_healpix_29'].iloc[0]}.npy", allow_pickle=True)
print("dtype:", obj.dtype, "shape:", obj.shape)

item = obj.item() if obj.shape == () else obj[0]
print("type:", type(item))
if isinstance(item, dict):
    for k, v in item.items():
        print(f"  {k}: {type(v).__name__}", np.shape(v))
else:
    print("len:", len(item), "first:", type(item[0]), np.shape(item[0]))

In [ ]:
# Simply viewing a sample
# ruff:noqa
import glob
import numpy as np
import matplotlib.pyplot as plt

files = sorted(glob.glob("cutouts/*.npy"))[:10]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, f in zip(axes.ravel(), files):
    g, r, z = np.asarray(np.load(f, allow_pickle=True).item()["flux"], dtype=np.float32)

    rgb = np.stack([g, r, z], axis=-1)
    rgb = np.arcsinh(rgb / np.percentile(rgb, 99.5))
    rgb = np.clip(rgb / rgb.max(), 0, 1)

    ax.imshow(rgb, origin="lower")
    ax.set_title(f.split("/")[-1][:12], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import time
import lsdb
import pandas as pd
from lsdb.streams import CatalogStream

N_TARGETS = None
RADIUS_ARCSEC = 1.0

t0 = time.time()
def log(msg):
    print(f"[{time.time()-t0:6.1f}s] {msg}", flush=True)

targets = pd.read_csv("targets_for_crossmatch.csv")
if N_TARGETS:
    targets = targets.head(N_TARGETS)
log(f"loaded {len(targets)} targets")

target_cat = lsdb.from_dataframe(
    targets,
    ra_column="ra_in",
    dec_column="dec_in",
    margin_threshold=5.0,
)
log("built target catalog")

legacy = lsdb.open_catalog(
    "hf://datasets/UniverseTBD/mmu_ssl_legacysurvey_north",
    columns=["object_id", "ra", "dec"],
)
log(f"opened legacy catalog — {len(legacy.get_healpix_pixels())} partitions total")

xm = target_cat.crossmatch(legacy, radius_arcsec=RADIUS_ARCSEC, n_neighbors=1)
log(f"crossmatch plan built — {len(xm.get_healpix_pixels())} partitions to scan")

batch, n_chunks, n_matches = [], 0, 0
for chunk in CatalogStream(catalog=xm):
    n_chunks += 1
    if len(chunk):
        batch.append(chunk)
        n_matches += len(chunk)
    log(f"chunk {n_chunks}: +{len(chunk)} rows (total {n_matches})")

log("stream finished")

matches = pd.concat(batch).reset_index() if batch else pd.DataFrame()
if len(matches):
    matches.to_csv("literature_x_legacysurvey_matches.csv", index=False)
    print(f"\n{len(matches)} / {len(targets)} targets matched ({len(matches)/len(targets):.1%})")
    print(matches["_dist_arcsec"].describe())
else:
    print("\nno matches")

/Users/gaurav/Desktop/Astrobridge/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[   0.0s] loaded 1772 targets
[   2.4s] built target catalog
[  12.1s] opened legacy catalog — 5488 partitions total


/Users/gaurav/Desktop/Astrobridge/env/lib/python3.13/site-packages/lsdb/catalog/catalog.py:410: FutureWarning: The default suffix behavior will change from applying suffixes to all columns to only applying suffixes to overlapping columns in a future release.To maintain the current behavior, explicitly set `suffix_method='all_columns'`. To change to the new behavior, set `suffix_method='overlapping_columns'`.
  warnings.warn(


[  13.3s] crossmatch plan built — 439 partitions to scan
[  70.9s] chunk 1: +1 rows (total 1)
[ 112.0s] chunk 2: +0 rows (total 1)
[ 151.3s] chunk 3: +1 rows (total 2)
[ 273.2s] chunk 4: +0 rows (total 2)
[ 292.0s] chunk 5: +1 rows (total 3)
[ 310.1s] chunk 6: +1 rows (total 4)
[ 342.4s] chunk 7: +0 rows (total 4)
[ 387.9s] chunk 8: +0 rows (total 4)
[ 427.2s] chunk 9: +0 rows (total 4)
[ 465.5s] chunk 10: +0 rows (total 4)
[ 506.4s] chunk 11: +1 rows (total 5)
[ 553.1s] chunk 12: +1 rows (total 6)
[ 590.2s] chunk 13: +1 rows (total 7)
[ 626.9s] chunk 14: +1 rows (total 8)
[ 666.7s] chunk 15: +1 rows (total 9)
[ 706.7s] chunk 16: +1 rows (total 10)
[ 761.2s] chunk 17: +1 rows (total 11)
[ 800.4s] chunk 18: +0 rows (total 11)
[ 834.5s] chunk 19: +2 rows (total 13)
[ 866.5s] chunk 20: +0 rows (total 13)
[ 906.9s] chunk 21: +1 rows (total 14)
[ 944.6s] chunk 22: +0 rows (total 14)
[ 965.6s] chunk 23: +0 rows (total 14)


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/UniverseTBD/mmu_ssl_legacysurvey_north/resolve/main/mmu_ssl_legacysurvey_north/dataset/Norder%3D6/Dir%3D10000/Npix%3D11514.parquet
Retrying in 1s [Retry 1/5].


[1139.1s] chunk 24: +1 rows (total 15)


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/UniverseTBD/mmu_ssl_legacysurvey_north/resolve/main/mmu_ssl_legacysurvey_north_10arcs/dataset/Norder%3D6/Dir%3D0/Npix%3D6529.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/UniverseTBD/mmu_ssl_legacysurvey_north/resolve/main/mmu_ssl_legacysurvey_north/dataset/Norder%3D6/Dir%3D0/Npix%3D6529.parquet
Retrying in 1s [Retry 1/5].


[1358.1s] chunk 25: +1 rows (total 16)
[1381.9s] chunk 26: +0 rows (total 16)
[1427.3s] chunk 27: +0 rows (total 16)
[1452.2s] chunk 28: +2 rows (total 18)
[1495.4s] chunk 29: +0 rows (total 18)
[1551.5s] chunk 30: +1 rows (total 19)
[1610.1s] chunk 31: +0 rows (total 19)
[1647.9s] chunk 32: +1 rows (total 20)
[1701.3s] chunk 33: +0 rows (total 20)
[1753.6s] chunk 34: +1 rows (total 21)
[1799.0s] chunk 35: +0 rows (total 21)
[1841.3s] chunk 36: +0 rows (total 21)
[1908.0s] chunk 37: +1 rows (total 22)
[1945.7s] chunk 38: +1 rows (total 23)
[2001.0s] chunk 39: +1 rows (total 24)
[2039.9s] chunk 40: +2 rows (total 26)
[2102.4s] chunk 41: +0 rows (total 26)
[2147.9s] chunk 42: +0 rows (total 26)
[2191.3s] chunk 43: +1 rows (total 27)
[2225.7s] chunk 44: +1 rows (total 28)
[2247.7s] chunk 45: +4 rows (total 32)
[2285.4s] chunk 46: +1 rows (total 33)
[2323.6s] chunk 47: +1 rows (total 34)
[2477.1s] chunk 48: +1 rows (total 35)
[2528.1s] chunk 49: +0 rows (total 35)
[2566.0s] chunk 50: +2 ro